# Refinery flares from NASA FIRMS (VIIRS / MODIS)

Detect refinery **flaring / fires** from satellite active-fire detections and plot a
per-refinery time series from **2010 to today**.

**Phase 1 (this notebook): NASA FIRMS only.**
- VIIRS 375 m active fire (S-NPP 2012+, NOAA-20 2018+) — *main focus*.
- MODIS 1 km active fire (2000+) — optional, fills 2010–2012.
- Refinery extents from a **GPKG** *or* a built-in sample of refineries that started
  operating **between 2012 and 2022** (so the flaring signal appears around start-up).

VIIRS active fire is not in the Planetary Computer STAC, so we use the FIRMS API
(the notebook still runs fine on the MPC JupyterHub). Black Marble night lights and
MPC raster products come in a later phase.

In [ ]:
import os, io, time
from datetime import date, timedelta
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

# ----------------------------- CONFIG -----------------------------
FIRMS_MAP_KEY = ""   # NASA FIRMS API key

# PRIMARY: analysis restricted to refinery POLYGONS from a GPKG.
# Set the path + name column; leave None to fall back to the built-in point sample.
REFINERIES_GPKG  = None          # e.g. "data/refineries.gpkg"
REFINERIES_LAYER = None          # layer name (None = first/only)
NAME_COL         = "name"        # column with the refinery name

# Ring around each feature (m). With GPKG POLYGONS we keep the analysis strictly inside
# the polygon (0 m). The point sample needs a ring to reach the flare stacks.
BUFFER_M  = 0 if REFINERIES_GPKG else 4000
START     = date(2010, 1, 1)     # graph starts here (VIIRS begins 2012; MODIS covers 2010-2012)
END       = date.today()

USE_VIIRS = True                 # FIRMS VIIRS 375 m  (S-NPP 2012+, NOAA-20 2018+)  <-- MAIN FOCUS
USE_MODIS = False                # FIRMS MODIS 1 km   (2000+)  — set True to add 2010-2012 (more download time)

CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)
print("GPKG:", REFINERIES_GPKG or "(sample)", "| buffer:", BUFFER_M, "m |", START, "->", END)

: 

## 1. Refineries
Built-in sample of refineries that **started operating between 2012 and 2022**. Centres
are pinned to each site's **actual VIIRS flare cluster** (located from 2020–2022
detections), so a tight buffer works. Replace with your own GPKG for exact boundaries.

In [ ]:
# Flare-stack centres located from VIIRS detection clusters (2020-2022), not guesses.
# name,                      lat,        lon,     start_year
SAMPLE = [
    ("SATORP Jubail (SA)",   27.0180,   49.5782,  2013),
    ("YASREF Yanbu (SA)",    23.9766,   38.2370,  2014),
    ("STAR Aliaga (TR)",     38.7394,   26.9459,  2018),
    ("Nghi Son (VN)",        19.3337,  105.7881,  2018),
    ("RAPID Pengerang (MY)",  1.3800,  104.1600,  2019),
    ("Jazan (SA)",           16.9390,   42.6328,  2018),
]

def load_refineries():
    if REFINERIES_GPKG:
        g = gpd.read_file(REFINERIES_GPKG, layer=REFINERIES_LAYER).to_crs(4326)
        if NAME_COL not in g.columns:
            g[NAME_COL] = [f"refinery_{i}" for i in range(len(g))]
        if "start_year" not in g.columns:
            g["start_year"] = np.nan
        return g[[NAME_COL, "start_year", "geometry"]].copy()
    return gpd.GeoDataFrame(
        {NAME_COL: [s[0] for s in SAMPLE], "start_year": [s[3] for s in SAMPLE]},
        geometry=[Point(s[2], s[1]) for s in SAMPLE], crs=4326)

ref = load_refineries()

# Buffered polygons (buffer in a metric CRS, then back to lon/lat).
ref_buf = ref.to_crs(3857).copy()
ref_buf["geometry"] = ref_buf.geometry.buffer(BUFFER_M)
ref_buf = ref_buf.to_crs(4326)

print(ref[[NAME_COL, "start_year"]].to_string(index=False))
ax = ref.plot(figsize=(11, 4), markersize=25, color="red")
ax.set_title("Refineries"); plt.show()

## 2. FIRMS download helpers
Area endpoint (**max 5 days / request**), chunked over the date range and **cached** per
chunk so re-runs are instant. Archive (`*_SP`) lags ~2 months; the recent window uses
near-real-time (`*_NRT`).

In [ ]:
FIRMS_URL = "https://firms.modaps.eosdis.nasa.gov/api/area/csv/{key}/{src}/{bbox}/{days}/{start}"
FIRMS_MAX_DAYS = 5   # FIRMS area endpoint allows at most 5 days per request

# Earliest date each source has data (skip pointless pre-launch requests).
SRC_MIN = {
    "MODIS_SP":        date(2000, 11, 1), "MODIS_NRT":        date(2000, 11, 1),
    "VIIRS_SNPP_SP":   date(2012,  1, 20), "VIIRS_SNPP_NRT":   date(2012,  1, 20),
    "VIIRS_NOAA20_SP": date(2018,  1,  1), "VIIRS_NOAA20_NRT": date(2018,  1,  1),
}

def firms_chunk(src, bbox, start, tag, days=FIRMS_MAX_DAYS):
    """One chunk (<=5 days) from FIRMS, cached on disk (empty results cached too)."""
    fn = os.path.join(CACHE_DIR, f"{tag}__{src}__{start}.csv".replace("/", "-").replace(" ", "_"))
    if os.path.exists(fn):
        try:
            return pd.read_csv(fn)
        except Exception:
            return pd.DataFrame()
    url = FIRMS_URL.format(key=FIRMS_MAP_KEY, src=src, bbox=bbox, days=days, start=start)
    r = requests.get(url, timeout=120); r.raise_for_status()
    txt = r.text.strip()
    first = txt.split("\n", 1)[0] if txt else ""
    if "," not in first:                       # not a CSV header -> message / error
        if "invalid" in txt.lower() or "error" in txt.lower():
            raise RuntimeError(txt[:200])
        df = pd.DataFrame()
    else:
        df = pd.read_csv(io.StringIO(txt))
    df.to_csv(fn, index=False)
    return df

def firms_download(sources, bbox, start, end, tag, days=FIRMS_MAX_DAYS, pause=0.15):
    frames = []
    for src in sources:
        d = max(start, SRC_MIN.get(src, start))
        while d <= end:
            try:
                df = firms_chunk(src, bbox, d.isoformat(), tag, days)
            except Exception as e:
                print(f"  ! {tag} {src} {d}: {e}"); df = pd.DataFrame()
            if len(df):
                df["source"] = src
                frames.append(df)
            d += timedelta(days=days)
            time.sleep(pause)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

## 3. Download active fire per refinery
Queried with a small per-refinery bbox (not one giant bbox). First run is slow
(~10–20 min for the sample); cached afterwards.

In [ ]:
assert FIRMS_MAP_KEY, "Set FIRMS_MAP_KEY (free: https://firms.modaps.eosdis.nasa.gov/api/area/)"

sources_sp  = ((['VIIRS_SNPP_SP', 'VIIRS_NOAA20_SP'] if USE_VIIRS else []) +
               (['MODIS_SP'] if USE_MODIS else []))
sources_nrt = ((['VIIRS_SNPP_NRT', 'VIIRS_NOAA20_NRT'] if USE_VIIRS else []) +
               (['MODIS_NRT'] if USE_MODIS else []))
nrt_start = END - timedelta(days=60)   # SP archive lags ~2 months; NRT for the recent window

all_fires = []
for _, row in ref_buf.iterrows():
    name = row[NAME_COL]
    minx, miny, maxx, maxy = row.geometry.bounds
    bbox = f"{minx:.4f},{miny:.4f},{maxx:.4f},{maxy:.4f}"
    print(f"== {name}  bbox={bbox}")
    a = firms_download(sources_sp,  bbox, START,     nrt_start - timedelta(days=1), tag=name)
    b = firms_download(sources_nrt, bbox, nrt_start, END,                           tag=name)
    df = pd.concat([x for x in (a, b) if len(x)], ignore_index=True) if (len(a) or len(b)) else pd.DataFrame()
    print(f"   -> {len(df)} raw detections in bbox")
    if len(df):
        all_fires.append(df)

fires = pd.concat(all_fires, ignore_index=True) if all_fires else pd.DataFrame()
print("TOTAL raw detections:", len(fires))

## 4. Keep detections inside the refinery buffers
Spatial join (point-in-polygon) drops bbox-corner false positives and tags each
detection with its refinery.

In [ ]:
fires["acq_date"] = pd.to_datetime(fires["acq_date"])
fpts = gpd.GeoDataFrame(
    fires, geometry=gpd.points_from_xy(fires.longitude, fires.latitude), crs=4326)

hits = gpd.sjoin(
    fpts, ref_buf[[NAME_COL, "start_year", "geometry"]],
    how="inner", predicate="within").rename(columns={NAME_COL: "refinery"})

print(len(hits), "detections inside refinery buffers")
hits[["refinery", "acq_date", "frp", "confidence", "daynight", "source"]].head()

In [ ]:
# Per-refinery summary: first detection vs. reported start year.
summary = (hits.groupby("refinery")
                .agg(first_detection=("acq_date", "min"),
                     last_detection=("acq_date", "max"),
                     total_detections=("acq_date", "size"),
                     fire_days=("acq_date", "nunique"))
                .reset_index()
                .merge(ref[[NAME_COL, "start_year"]].rename(columns={NAME_COL: "refinery"}),
                       on="refinery", how="left"))
summary

## 5. Time series (2010 → now)
Monthly detections per refinery, with a dashed line at the reported start year — the
flaring signal should switch on around start-up.

In [ ]:
hits["month"] = hits["acq_date"].values.astype("datetime64[M]")
monthly = (hits.groupby(["refinery", "month"])
                .agg(detections=("acq_date", "size"))
                .reset_index())

refs = sorted(monthly["refinery"].unique())
n = len(refs)
fig, axes = plt.subplots(n, 1, figsize=(13, 2.3 * n), sharex=True)
axes = np.atleast_1d(axes)
for ax, name in zip(axes, refs):
    g = monthly[monthly.refinery == name]
    ax.bar(g["month"], g["detections"], width=18, color="orangered")
    sy = ref.loc[ref[NAME_COL] == name, "start_year"]
    if len(sy) and pd.notna(sy.iloc[0]):
        x0 = pd.Timestamp(int(sy.iloc[0]), 1, 1)
        ax.axvline(x0, color="navy", ls="--", lw=1)
        ax.text(x0, ax.get_ylim()[1] * 0.8, f"  start {int(sy.iloc[0])}", color="navy", fontsize=8)
    ax.set_title(name, fontsize=9, loc="left")
    ax.set_ylabel("det./mo")
    ax.set_xlim(pd.Timestamp(START), pd.Timestamp(END))
axes[-1].set_xlabel("year")
fig.suptitle("Active-fire (VIIRS/MODIS) detections at refineries — monthly", y=1.001)
plt.tight_layout(); plt.show()

In [ ]:
# Optional: save the detections for reuse / GIS.
hits.drop(columns="index_right", errors="ignore").to_file(
    "refinery_fire_detections.gpkg", driver="GPKG")
monthly.to_csv("refinery_fire_monthly.csv", index=False)
print("saved refinery_fire_detections.gpkg + refinery_fire_monthly.csv")

## Next phases
- **Black Marble night lights** (VNP46) for radiance trends at each site.
- Planetary Computer raster products / MODIS FireMask cross-check.
- Flare-vs-incident separation (persistent low-FRP night detections = flaring;
  short bright spikes = incidents).